# 1. Introdução

Neste notebook utilizamos **Transfer Learning** com a rede **MobileNet** (alpha=0.25) para construímos um classificador binário de **detecção de pessoas** (PESSOA / NENHUMA).

O modelo resultante é ultra-leve (~217 KB em Float16, ~155 KB em INT8) e otimizado para execução em dispositivos embarcados como o Raspberry Pi.

**Pipeline:**
1. Carregar dataset COCO recortado (1000 imagens por classe)
2. Aplicar data augmentation com camadas Keras
3. Treinar camada classificadora com base congelada (80 epochs)
4. Fine-tuning com todas as camadas descongeladas (1 epoch)
5. Exportar para TFLite (Float16 + INT8)

# 2. Parâmetros

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Classes — classificação binária
CLASSES = ["NENHUM", "PESSOA"]
NUM_CLASSES = len(CLASSES)

# Caminhos
DATA_DIR = "data"       # dataset com subpastas NENHUM/ e PESSOA/
MODEL_DIR = "model"     # saída dos modelos exportados

# Arquitetura MobileNet
IMAGE_DIM = 96
MOBILENET_ALPHA = 0.25
CUT_LAYER = "conv_pw_10_relu"

# Treinamento — Fase 1 (base congelada)
FROZEN_EPOCHS = 80
FROZEN_LR = 0.0005

# Treinamento — Fase 2 (fine-tuning)
FINETUNE_EPOCHS = 1
FINETUNE_LR = 0.00001

# Parâmetros comuns
BATCH_SIZE = 100
DROPOUT_RATE = 0.1
VALIDATION_SPLIT = 0.2
RANDOM_SEED = 42

print(f"TensorFlow {tf.__version__}")
print(f"Classes: {CLASSES}")
print(f"Imagem: {IMAGE_DIM}x{IMAGE_DIM}x3")

# 3. Preparação dos Dados

## 3.1. Carga do Dataset

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(IMAGE_DIM, IMAGE_DIM),
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=RANDOM_SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(IMAGE_DIM, IMAGE_DIM),
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=RANDOM_SEED,
)

print(f"Classes encontradas: {train_ds.class_names}")
print(f"Batches treino: {len(train_ds)}, validação: {len(val_ds)}")

## 3.2. Data Augmentation

In [ ]:
# Camadas de augmentação (aplicadas apenas no treino)
augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.03),        # ~10°
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomTranslation(0.1, 0.1),
])

# Rescaling: [0, 255] → [0, 1]
rescaling = keras.layers.Rescaling(1.0 / 255)

# Aplicar augmentação + rescaling ao treino
train_ds = train_ds.map(
    lambda x, y: (rescaling(augmentation(x, training=True)), y),
    num_parallel_calls=tf.data.AUTOTUNE,
)

# Apenas rescaling na validação
val_ds = val_ds.map(
    lambda x, y: (rescaling(x), y),
    num_parallel_calls=tf.data.AUTOTUNE,
)

# Prefetch para performance
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print("Data augmentation aplicada ao dataset de treino.")

## 3.3. Visualização

In [ ]:
plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(CLASSES[labels[i]])
        plt.axis("off")
plt.suptitle("Amostras do Dataset (com augmentação)", fontsize=14)
plt.tight_layout()
plt.show()

# 4. Modelagem

## 4.1. Modelo Base (MobileNet)

In [ ]:
# Carregar MobileNet pré-treinada (ImageNet)
base_model = keras.applications.MobileNet(
    weights="imagenet",
    input_shape=(IMAGE_DIM, IMAGE_DIM, 3),
    alpha=MOBILENET_ALPHA,
    include_top=False,
)

# Congelar base para transfer learning
base_model.trainable = False

# Cortar na camada intermediária para modelo mais leve
last_layer = base_model.get_layer(CUT_LAYER)

# Classificador customizado
x = keras.layers.Reshape((-1, last_layer.output.shape[3]))(last_layer.output)
x = keras.layers.Dropout(DROPOUT_RATE)(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(base_model.input, x)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FROZEN_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

# 5. Treinamento (Base Congelada)

In [ ]:
history = model.fit(
    train_ds,
    epochs=FROZEN_EPOCHS,
    validation_data=val_ds,
    shuffle=True,
)

print(f"\nTreinamento base congelada concluído ({FROZEN_EPOCHS} epochs).")

## 5.1. Desempenho

In [ ]:
plt.style.use("ggplot")
plt.figure()
plt.plot(np.arange(FROZEN_EPOCHS), history.history["loss"], label="train_loss")
plt.plot(np.arange(FROZEN_EPOCHS), history.history["val_loss"], label="val_loss")
plt.title("Loss — Base Congelada")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(loc="upper right")
plt.show()

In [ ]:
plt.style.use("ggplot")
plt.figure()
plt.plot(np.arange(FROZEN_EPOCHS), history.history["accuracy"], label="accuracy")
plt.plot(np.arange(FROZEN_EPOCHS), history.history["val_accuracy"], label="val_accuracy")
plt.title("Acurácia — Base Congelada")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.show()

# 6. Fine Tuning

In [ ]:
# Descongelar todas as camadas para fine-tuning
base_model.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_ft = model.fit(
    train_ds,
    epochs=FINETUNE_EPOCHS,
    validation_data=val_ds,
    shuffle=True,
)

print(f"\nFine-tuning concluído ({FINETUNE_EPOCHS} epoch).")

## 6.1. Avaliação Final

In [ ]:
print("Avaliação no conjunto de validação:")
loss, accuracy = model.evaluate(val_ds)
print(f"\nLoss: {loss:.4f}")
print(f"Acurácia: {accuracy:.4f} ({accuracy*100:.2f}%)")

# 7. Conversão e Exportação

## 7.1. Modelo Keras

In [ ]:
import os
os.makedirs(MODEL_DIR, exist_ok=True)

keras_path = os.path.join(MODEL_DIR, "person-detect-model.keras")
model.save(keras_path)
print(f"Modelo Keras salvo em: {keras_path}")
print(f"Tamanho: {os.path.getsize(keras_path) / 1024:.0f} KB")

## 7.2. TFLite Float16

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_f16 = converter.convert()

f16_path = os.path.join(MODEL_DIR, "person-detect-model.tflite")
with open(f16_path, "wb") as f:
    f.write(tflite_f16)

print(f"TFLite Float16 salvo em: {f16_path}")
print(f"Tamanho: {os.path.getsize(f16_path) / 1024:.0f} KB")

## 7.3. TFLite INT8

In [ ]:
# Dataset representativo para calibração INT8
def representative_dataset_gen():
    for images, _ in train_ds.take(10):
        for img in images:
            yield [tf.expand_dims(img, axis=0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_gen
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.target_spec.supported_types = [tf.int8]
converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_int8 = converter_int8.convert()

int8_path = os.path.join(MODEL_DIR, "person-detect-model-int8.tflite")
with open(int8_path, "wb") as f:
    f.write(tflite_int8)

print(f"TFLite INT8 salvo em: {int8_path}")
print(f"Tamanho: {os.path.getsize(int8_path) / 1024:.0f} KB")

## 7.4. Avaliação do Modelo Convertido

In [ ]:
# Avaliar modelo Float16 TFLite no dataset de validação
interpreter = tf.lite.Interpreter(model_path=f16_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_index = input_details[0]["index"]
output_index = output_details[0]["index"]
input_dtype = input_details[0]["dtype"]

correct = 0
total = 0

for images, labels in val_ds:
    for i in range(len(labels)):
        img = tf.expand_dims(images[i], axis=0)
        img = tf.cast(img, input_dtype)
        interpreter.set_tensor(input_index, img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)
        pred = np.argmax(output[0])
        if pred == labels[i].numpy():
            correct += 1
        total += 1

tflite_accuracy = correct / total
print(f"Avaliados {total} exemplos.")
print(f"Acurácia TFLite Float16: {tflite_accuracy:.4f} ({tflite_accuracy*100:.2f}%)")